# FIFA World Cup 2026 — Notebook 02: Team Strengths

## About

**Purpose:** Turn the clean match-results table into an attack strength and a defence strength for every team.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-05<br>
**Notes:** Built in two passes. First a *plain* version (all matches weighted equally) to expose the small-sample problem and the shrinkage fix. Then a *recency-weighted* version so current form outweighs historical reputation — the strengths that actually feed the model.<br>
**Description:** Reads `played_matches.parquet` (notebook 01) and produces a per-team strength table: attack (goals scored vs. an average team) and defence (goals conceded vs. average). Saved to `team_strengths.parquet` for the Poisson scoreline model in notebook 03.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-06-05 | 1.0     | Ganapathy K | Initial version |

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
from pathlib import Path

### 1.2 Config

Paths plus the two tunable knobs: `SHRINKAGE_GAMES` (how hard to distrust small samples) and `HALF_LIFE_YEARS` (how fast old matches lose weight).

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
PLAYED_MATCHES_PATH = PROCESSED_DATA_DIR / "played_matches.parquet"
TEAM_STRENGTHS_PATH = PROCESSED_DATA_DIR / "team_strengths.parquet"

SHRINKAGE_GAMES = 30      # phantom games at league-average added to every team
HALF_LIFE_YEARS = 4       # a match this old counts half as much as a brand-new one

## 2. Load Played Matches

Read the clean played-matches table that notebook 01 saved as parquet. Parquet preserves dtypes, so the scores come back as `int` with no re-parsing.

In [4]:
played_matches = pd.read_parquet(PLAYED_MATCHES_PATH)
print(f"Loaded {len(played_matches)} played matches")
played_matches.head()

Loaded 49306 played matches


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


## 3. Team Strengths — Plain Version

First build strengths treating every match equally. This exposes the small-sample problem and the shrinkage fix before we add recency in section 4.

### 3.1 Baseline — average goals per game

The reference point: how many goals does an *average* team score in an *average* game? Every match has two scoring sides, so the number of team-game slots is `2 × number of matches`. All strengths are measured relative to this number.

In [5]:
total_goals = played_matches["home_score"].sum() + played_matches["away_score"].sum()
team_game_slots = 2 * len(played_matches)
baseline_goals_per_game = total_goals / team_game_slots

print(f"Total goals scored:        {total_goals}")
print(f"Team-game slots:           {team_game_slots}")
print(f"Baseline goals per game:   {baseline_goals_per_game:.3f}")

Total goals scored:        144940
Team-game slots:           98612
Baseline goals per game:   1.470


### 3.2 Reshape to one row per team

Each match currently has two teams side by side. To group by team, split every match into two rows — one from each team's point of view — recording goals scored and goals conceded. The 49,306 matches become 98,612 team-rows. `date` is kept for the recency step later.

In [6]:
home_view = played_matches[["date", "home_team", "home_score", "away_score"]].rename(
    columns={"home_team": "team", "home_score": "goals_scored", "away_score": "goals_conceded"}
)
away_view = played_matches[["date", "away_team", "away_score", "home_score"]].rename(
    columns={"away_team": "team", "away_score": "goals_scored", "home_score": "goals_conceded"}
)

team_matches = pd.concat([home_view, away_view], ignore_index=True)

print(f"Match rows:      {len(played_matches)}")
print(f"Team rows:       {len(team_matches)}")
team_matches.head()

Match rows:      49306
Team rows:       98612


,date,team,goals_scored,goals_conceded
0,1872-11-30,Scotland,0,0
1,1873-03-08,England,4,2
2,1874-03-07,Scotland,2,1
3,1875-03-06,England,2,2
4,1876-03-04,Scotland,3,0


### 3.3 Per-team averages

Group by `team` and take the mean of `goals_scored` and `goals_conceded` — each team's raw scoring and conceding rate per game. Also count games played. Sorting by raw scoring shows the problem: tiny-sample teams post freak averages.

In [7]:
team_averages = team_matches.groupby("team").agg(
    games_played=("goals_scored", "size"),
    avg_goals_scored=("goals_scored", "mean"),
    avg_goals_conceded=("goals_conceded", "mean"),
)

print(f"Teams: {len(team_averages)}")
team_averages.sort_values("avg_goals_scored", ascending=False).head(10)

Teams: 336


,games_played,avg_goals_scored,avg_goals_conceded
team,,,
Quebec,3,8.000000,0.666667
Elba Island,1,5.000000,0.000000
Yorkshire,7,3.857143,1.571429
Parishes of Jersey,3,3.666667,1.333333
Cascadia,7,3.285714,2.000000
Isle of Man,58,3.206897,1.500000
Provence,23,3.173913,2.565217
Occitania,33,3.121212,1.151515
Sápmi,29,3.103448,1.862069


### 3.4 Strengths with shrinkage

Convert averages to strength ratios (÷ baseline), but first *shrink* each average toward the baseline using `SHRINKAGE_GAMES` phantom games at league-average:

```
shrunk = (games × team_average + K × baseline) / (games + K)
```

A team with 3 games gets dragged almost fully back to average; a team with 1,000 games barely moves. This is regularization — don't trust thin evidence.

In [8]:
plain_strengths = team_averages.copy()

shrunk_scored = (
    plain_strengths["games_played"] * plain_strengths["avg_goals_scored"]
    + SHRINKAGE_GAMES * baseline_goals_per_game
) / (plain_strengths["games_played"] + SHRINKAGE_GAMES)

shrunk_conceded = (
    plain_strengths["games_played"] * plain_strengths["avg_goals_conceded"]
    + SHRINKAGE_GAMES * baseline_goals_per_game
) / (plain_strengths["games_played"] + SHRINKAGE_GAMES)

plain_strengths["attack_strength"] = shrunk_scored / baseline_goals_per_game
plain_strengths["defence_strength"] = shrunk_conceded / baseline_goals_per_game

plain_strengths.sort_values("attack_strength", ascending=False).head(10)

,games_played,avg_goals_scored,avg_goals_conceded,attack_strength,defence_strength
team,,,,,
Isle of Man,58,3.206897,1.500000,1.778952,1.013542
Jersey,235,2.744681,1.212766,1.769189,0.844920
Tahiti,242,2.714876,1.533058,1.753674,1.038291
New Caledonia,265,2.633962,1.392453,1.711506,0.952727
Guernsey,240,2.566667,1.304167,1.663350,0.899830
Occitania,33,3.121212,1.151515,1.588532,0.886569
Sápmi,29,3.103448,1.862069,1.546318,1.131181
Germany,1030,2.250485,1.162136,1.516117,0.796600
Northern Cyprus,34,2.882353,0.911765,1.510558,0.798301


## 4. Recency Weighting

The plain version still treats a 1970 result like a 2025 one, so it encodes reputation, not current form. Fix it by weighting each match by how recent it is, then recomputing strengths.

### 4.1 Weight each match by recency

Exponential time-decay: weight = `0.5 ** (age_in_years / HALF_LIFE_YEARS)`. A brand-new match weighs ~1.0; a match one half-life old weighs 0.5; two half-lives, 0.25. `reference_date` is the most recent match in the data.

In [9]:
reference_date = team_matches["date"].max()
age_years = (reference_date - team_matches["date"]).dt.days / 365.25
team_matches["weight"] = 0.5 ** (age_years / HALF_LIFE_YEARS)

print(f"Reference date: {reference_date.date()}")
team_matches[["date", "team", "goals_scored", "goals_conceded", "weight"]].head()

Reference date: 2026-06-03


,date,team,goals_scored,goals_conceded,weight
0,1872-11-30,Scotland,0,0,2.803768e-12
1,1873-03-08,England,4,2,2.937206e-12
2,1874-03-07,Scotland,2,1,3.490875e-12
3,1875-03-06,England,2,2,4.148913e-12
4,1876-03-04,Scotland,3,0,4.930991e-12


### 4.2 Recency-weighted strengths

Replace plain averages with *weighted* ones. For each team, sum `weight × goals` and divide by the team's total weight — recent games dominate. Shrinkage still applies, now in effective-games units (`weight_sum`). The league baseline is also recomputed weighted, so numerator and denominator use the same recent window.

In [10]:
team_matches["weighted_scored"] = team_matches["weight"] * team_matches["goals_scored"]
team_matches["weighted_conceded"] = team_matches["weight"] * team_matches["goals_conceded"]

weighted_baseline = team_matches["weighted_scored"].sum() / team_matches["weight"].sum()

weighted = team_matches.groupby("team").agg(
    games_played=("weight", "size"),
    weight_sum=("weight", "sum"),
    weighted_scored_sum=("weighted_scored", "sum"),
    weighted_conceded_sum=("weighted_conceded", "sum"),
)

weighted["attack_strength"] = (
    (weighted["weighted_scored_sum"] + SHRINKAGE_GAMES * weighted_baseline)
    / (weighted["weight_sum"] + SHRINKAGE_GAMES)
) / weighted_baseline

weighted["defence_strength"] = (
    (weighted["weighted_conceded_sum"] + SHRINKAGE_GAMES * weighted_baseline)
    / (weighted["weight_sum"] + SHRINKAGE_GAMES)
) / weighted_baseline

team_strengths = weighted[["games_played", "attack_strength", "defence_strength"]]

print(f"Weighted baseline: {weighted_baseline:.3f}  (global was {baseline_goals_per_game:.3f})")
team_strengths[team_strengths["games_played"] >= 100].sort_values("attack_strength", ascending=False).head(15)

Weighted baseline: 1.370  (global was 1.470)


,games_played,attack_strength,defence_strength
team,,,
Spain,781,1.532488,0.692586
Japan,790,1.481497,0.699387
Germany,1030,1.475110,0.868762
Portugal,693,1.460969,0.715147
Belgium,852,1.446350,0.775606
Netherlands,878,1.445932,0.798409
England,1088,1.374522,0.642319
Algeria,616,1.373497,0.694926
France,933,1.362550,0.735176


### 4.3 Sanity check — plain vs recency

Compare attack strength before and after recency for the traditional powers. Faded giants should drop, recent risers should climb.

In [11]:
big_teams = ["Brazil", "France", "Spain", "Germany", "Argentina",
             "England", "Italy", "Belgium", "Portugal", "Netherlands"]

comparison = pd.DataFrame({
    "plain_attack": plain_strengths["attack_strength"],
    "recency_attack": team_strengths["attack_strength"],
}).loc[big_teams]
comparison["change"] = comparison["recency_attack"] - comparison["plain_attack"]
comparison.round(3)

,plain_attack,recency_attack,change
team,,,
Brazil,1.468,1.336,-0.131
France,1.240,1.363,0.123
Spain,1.378,1.532,0.155
Germany,1.516,1.475,-0.041
Argentina,1.282,1.344,0.062
England,1.473,1.375,-0.099
Italy,1.187,1.235,0.048
Belgium,1.226,1.446,0.221
Portugal,1.192,1.461,0.269


## 5. Save

Write the recency-weighted strengths table to `data/processed/` for notebook 03 (Poisson scoreline model).

In [12]:
team_strengths.to_parquet(TEAM_STRENGTHS_PATH)
print(f"Saved {len(team_strengths)} team strengths \u2192 {TEAM_STRENGTHS_PATH}")

Saved 336 team strengths → D:\Data Science\Visual Studio Code\fifa_wc_2026_poisson\data\processed\team_strengths.parquet
